In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense
from tensorflow.keras.utils import to_categorical

In [ ]:
# CSV読み込み
train_df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
test_df = pd.read_csv('/kaggle/input/digit-recognizer/test.csv')

# 画像とラベルを分ける
train_images = train_df.drop('label', axis=1).values / 255.0  # 0-1正規化
train_labels = train_df['label'].values
test_images = test_df.values / 255.0  # 0-1正規化

# CNN用に形を変更（28x28x1）
X_train_cnn = train_images.reshape(-1,28,28,1)
X_test_cnn = test_images.reshape(-1,28,28,1)

# ラベルをone-hotエンコーディング
y_train_cat = to_categorical(train_labels, 10)

In [ ]:
from tensorflow.keras import models, layers, initializers

In [ ]:
from tensorflow.keras import models, layers, initializers

model = models.Sequential()

# 入力層：28×28の画像を1チャンネルに整形
model.add(layers.Reshape((28, 28, 1), input_shape=(28*28,), name='reshape'))

# 畳み込み層①
model.add(layers.Conv2D(
    32, (3, 3), padding='same',
    kernel_initializer=initializers.TruncatedNormal(stddev=0.1),
    activation='relu',
    name='conv_filter1'
))

# プーリング層①
model.add(layers.MaxPooling2D(pool_size=(2, 2), name='pool1'))

# 畳み込み層②
model.add(layers.Conv2D(
    128, (5, 5), padding='same',
    kernel_initializer=initializers.TruncatedNormal(stddev=0.1),
    activation='relu',
    name='conv_filter2'
))

# プーリング層②
model.add(layers.MaxPooling2D(pool_size=(2, 2), name='pool2'))

# 平坦化
model.add(layers.Flatten(name='flatten'))

# 全結合層
model.add(layers.Dense(
    1024, activation='relu',
    kernel_initializer=initializers.TruncatedNormal(stddev=0.1),
    name='hidden'
))

# ドロップアウト層（過学習防止）
model.add(layers.Dropout(rate=0.5, name='dropout'))

# 出力層
model.add(layers.Dense(10, activation='softmax', name='softmax'))

# モデル構造の確認
model.summary()

In [ ]:
# --- モデルのコンパイル ---
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # one-hotなのでcategorical_crossentropyを使う
    metrics=['accuracy']
)

# --- モデルの学習 ---
history = model.fit(
    train_images, y_train_cat,
    epochs=30,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

# --- 学習の様子を確認 ---
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.title('Accuracy')
plt.legend()
plt.show()
"""
# --- 推論（提出用） ---
pred = model.predict(X_test_)
pred_labels = pred.argmax(axis=1)

submission = pd.DataFrame({
    'ImageId': range(1, len(pred_labels) + 1),
    'Label': pred_labels
})
submission.to_csv('submission.csv', index=False)
print("✅ submission.csv を作成しました！")
"""

In [ ]:
preds = model.predict(test_images)
preds_classes = np.argmax(preds, axis=1)  # 確率の最大値を整数ラベルに変換

In [ ]:
# ===== 提出ファイル作成 =====
submission = pd.DataFrame({
    "ImageId": np.arange(1, len(preds)+1),
    "Label": preds_classes
})
submission_path = "/kaggle/working/submission.csv"
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)
submission.head()